In [ ]:
# =========================================================
# UNSLOTH + MODEL
# =========================================================

import torch

# UNSLOTH
!pip install -U unsloth[colab-new]

# Yardımcı araçlar
!pip install fastapi uvicorn nest_asyncio python-multipart pillow PyPDF2 python-docx

# Cloudflare
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# MODEL YÜKLEME
from unsloth import FastLanguageModel
from google.colab import drive
from transformers import ViTImageProcessor, ViTForImageClassification

drive.mount('/content/drive')

model, tokenizer = FastLanguageModel.from_pretrained(
    "/content/drive/My Drive/Final_Project_Model",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
print("Dil modeli yüklendi")

vit_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
vit_model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")

print("TÜM SİSTEM TEMİZ VE HAZIR")


In [2]:
import nest_asyncio
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading
import subprocess
import time
import re
import json
import ast
import io
from PyPDF2 import PdfReader
import docx
from PIL import Image
import torch

# API Kurulumu
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- MODEL SINIFLARI ---
class CVRequest(BaseModel):
    text: str

class MatchRequest(BaseModel):
    cv_text: str
    job_desc: str

class EmailRequest(BaseModel):
    ad_soyad: str
    pozisyon: str

class ChatRequest(BaseModel):
    cv_text: str
    question: str

# --- YARDIMCI DOSYA OKUMA FONKSİYONLARI ---

def extract_text_from_pdf(file_bytes):
    try:
        reader = PdfReader(io.BytesIO(file_bytes))
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text.strip()
    except:
        return ""

def extract_text_from_docx(file_bytes):
    try:
        doc = docx.Document(io.BytesIO(file_bytes))
        text = "\n".join([para.text for para in doc.paragraphs])
        return text.strip()
    except:
        return ""

# --- JSON KURTARICI ---
def clean_and_parse_json(text):
    """
    Modelin ürettiği metni (tek tırnaklı, Python dict, bozuk JSON)
    ne olursa olsun geçerli bir objeye çevirir.
    """
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if not match: return None
        json_str = match.group()
        json_str = json_str.replace("True", "true").replace("False", "false").replace("None", "null")
        try: return ast.literal_eval(json_str)
        except: pass
        try: return json.loads(json_str.replace("'", '"'))
        except: return None
    except: return None

# --- ANA ANALİZ FONKSİYONLARI ---

def cv_to_json_internal(cv_text):
    prompt = f"""### Instruction:
You are an expert HR AI. Analyze the resume below.
Extract these fields into a valid JSON object:
- "ad_soyad": String (Name)
- "eposta": String (Email)
- "egitim_durumu": String (Education Level)
- "yetenekler": List of Strings (Technical Skills only)
- "toplam_deneyim_yili": Integer (Total years)

### Input CV:
{cv_text[:2000]}

### Response (JSON Only):
"""
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
    sonuc = tokenizer.batch_decode(outputs)[0]
    ham_metin = sonuc.split("### Response:")[-1].replace("<|end_of_text|>", "").strip()

    final_data = clean_and_parse_json(ham_metin)
    if not final_data: final_data = {"hata": "JSON bulunamadı"}

    if "hata" not in final_data:
        skill_count = len(final_data.get("yetenekler", []))
        exp_years = final_data.get("toplam_deneyim_yili", 0)
        try: exp_years = int(exp_years)
        except: exp_years = 0

        edu = str(final_data.get("egitim_durumu", "")).lower()
        edu_score = 60
        if any(x in edu for x in ["lisans", "bachelor", "bs", "fakülte", "mühendisliği"]): edu_score = 80
        if any(x in edu for x in ["yüksek", "master", "ms", "mba"]): edu_score = 90
        if any(x in edu for x in ["doktora", "phd"]): edu_score = 100

        final_data["analiz_puanlari"] = {
            "Technical": min(skill_count * 12, 100),
            "Experience": min(exp_years * 20, 100),
            "Education": edu_score,
            "Soft Skills": 75,
            "Completeness": 90
        }
    return {"json_cikti": final_data}

# --- ENDPOINTLER ---

@app.post("/chat_cv")
async def chat_cv(data: ChatRequest):
    try:
        prompt = f"""### Instruction:
Answer based ONLY on the CV below. Keep it short.

### CV:
{data.cv_text[:2500]}

### Question:
{data.question}

### Answer:
"""
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.5)
        sonuc = tokenizer.batch_decode(outputs)[0]
        answer = sonuc.split("### Answer:")[-1].replace("<|end_of_text|>", "").strip()
        return {"answer": answer if answer else "No response."}
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

@app.post("/match")
async def match_job(data: MatchRequest):
    # Job Match Prompt
    prompt = f"""### Instruction:
Act as a strict AI Recruiter. Compare the Candidate CV with the Job Description.

CRITICAL SCORING RULES:
1. MISMATCH PENALTY: If the Job Title (e.g., 'Grocer', 'Driver', 'Sales') has NOTHING to do with the CV (e.g., 'Engineer', 'Developer'), the score MUST be between 0 and 20.
2. PARTIAL MATCH: If skills match but experience is low, score between 40-60.
3. PERFECT MATCH: Only give 80+ if titles and skills align perfectly.

Candidate CV:
{data.cv_text[:800]}

Job Description:
{data.job_desc[:400]}

Return a Python dictionary: {{'score': <0-100>, 'reason': '<Short explanation>'}}

### Response:
"""
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1)
    sonuc = tokenizer.batch_decode(outputs)[0]
    ham_metin = sonuc.split("### Response:")[-1].replace("<|end_of_text|>", "").strip()

    cleaned_data = clean_and_parse_json(ham_metin)
    if cleaned_data: return cleaned_data
    else: return {"score": 0, "reason": "Analiz başarısız."}

@app.post("/write_email")
async def write_email(data: EmailRequest):
    # Prompt Sızmasını (Leakage) Engelleme

    prompt = f"""### Instruction:
Ignore all previous instructions. You are a professional HR Manager.
Task: Write a formal interview invitation email in Turkish.
Context: You are inviting {data.ad_soyad} for the position of {data.pozisyon}.

RULES:
- Do NOT include any JSON data.
- Do NOT act as the candidate.
- Do NOT make up fake experience details.
- Write ONLY the email body.

### Response:
Konu: İş Görüşmesi Daveti - {data.pozisyon}

Sayın {data.ad_soyad},

"""
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    # Temperature 0.6
    outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.6)
    sonuc = tokenizer.batch_decode(outputs)[0]

    email_body = sonuc.split("### Response:")[-1].replace("<|end_of_text|>", "").strip()


    blacklist = ["json", "formatına", "dönüştür", "{", "}", "none", "tecrübene sahip ol", "lise mezunu ol"]
    is_garbage = any(word in email_body.lower() for word in blacklist)

    if is_garbage or len(email_body) < 30:
        print(" Model Çalışıyor!")
        email_body = (
            f"Konu: İş Görüşmesi Daveti - {data.pozisyon}\n\n"
            f"Sayın {data.ad_soyad},\n\n"
            f"{data.pozisyon} pozisyonu için başvurunuzu inceledik. Niteliklerinizin şirketimiz için uygun olabileceğini düşünüyoruz.\n\n"
            f"Sizi daha yakından tanımak ve tecrübeleriniz üzerine konuşmak için bir görüşmeye davet etmek isteriz.\n\n"
            f"Müsait olduğunuz zaman dilimlerini bizimle paylaşırsanız seviniriz.\n\n"
            f"Saygılarımızla,\nİnsan Kaynakları Ekibi"
        )
    else:

        if "Konu:" not in email_body:
             email_body = f"Konu: İş Görüşmesi Daveti - {data.pozisyon}\n\nSayın {data.ad_soyad},\n\n" + email_body

    return {"email": email_body}

@app.post("/cevir")
async def cv_to_json(data: CVRequest):
    result = cv_to_json_internal(data.text)
    result["raw_text"] = data.text
    return result

@app.post("/cevir_dosya")
async def cv_dosya_analiz(file: UploadFile = File(...)):
    try:
        contents = await file.read()
        filename = file.filename.lower()
        if filename.endswith('.pdf'): text = extract_text_from_pdf(contents)
        elif filename.endswith('.docx'): text = extract_text_from_docx(contents)
        else: return {"json_cikti": {"hata": "Format desteklenmiyor"}}

        result = cv_to_json_internal(text)
        result["raw_text"] = text
        return result
    except Exception as e: return {"json_cikti": {"hata": str(e)}}

@app.post("/analiz_foto")
async def analyze_photo(file: UploadFile = File(...)):
    try:
        image = Image.open(io.BytesIO(await file.read())).convert("RGB")
        inputs = vit_processor(images=image, return_tensors="pt")
        outputs = vit_model(**inputs)
        idx = outputs.logits.argmax(-1).item()
        confidence = torch.nn.functional.softmax(outputs.logits, dim=-1)[0][idx].item()
        label = vit_model.config.id2label[idx].lower()

        # Profesyonel anahtar kelimeler
        pro_keywords = [
            'suit', 'tie', 'business', 'formal', 'jacket', 'tuxedo', 'groom',
            'uniform', 'academic_gown', 'bow_tie', 'shirt', 'official',
            'collared', 'portrait', 'person', 'man', 'woman'
        ]

        is_pro = any(k in label for k in pro_keywords)
        return {
            "sonuc": "Başarılı",
            "kategori": "Professional" if is_pro else "Casual",
            "detay": label,
            "skor": int(confidence * 100)
        }
    except Exception as e:
        return {"sonuc": "Hata", "mesaj": str(e)}
# --- MÜLAKAT SORUSU OLUŞTURUCU ---
@app.post("/generate_questions")
async def generate_interview_questions(data: MatchRequest):

    prompt = f"""### Instruction:
    Ignore previous tasks. Act as a Technical Interviewer.
    Based on the text below, ask 3 technical interview questions.
    Do NOT summarize the CV. Do NOT act as the candidate.
    ONLY write questions ending with '?'.

    Text:
    {data.cv_text[:500]}

    ### Response:
    1. [Teknik] Soru:"""

    try:
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.8)
        sonuc = tokenizer.batch_decode(outputs)[0]
        questions = sonuc.split("### Response:")[-1].replace("<|end_of_text|>", "").strip()


        if not questions.startswith("1. [Teknik]"):
            questions = "1. [Teknik] Soru: " + questions

    except:
        questions = ""



    lower_q = questions.lower()
    bad_keywords = ["mezunuyum", "mail yok", "mail adresim", "tecrübem var", "biliyorum", "ön lisans", "lise"]

    # Eğer soru çok kısaysa
    if any(k in lower_q for k in bad_keywords) or "?" not in questions or len(questions) < 50:
        print(" Model Çalışıyor!")

        # CV ve İlan Metnini Analiz Et
        combined_text = (data.cv_text + " " + data.job_desc).lower()

        # Soru Havuzu
        pool = []

        # Python
        if "python" in combined_text:
            pool.append("1. [Teknik] Soru: Python'da 'list' ve 'tuple' arasındaki temel farklar nelerdir ve hangisi hangi durumda tercih edilmelidir?")
            pool.append("2. [Teknik] Soru: Python'da memory management (bellek yönetimi) ve Garbage Collector nasıl çalışır?")

        # Java
        if "java" in combined_text:
            pool.append("1. [Teknik] Soru: Java'da OOP prensiplerini (Encapsulation, Inheritance, Polymorphism) gerçek bir proje üzerinden örneklendirebilir misiniz?")
            pool.append("2. [Teknik] Soru: Java Spring Boot projelerinde Dependency Injection tam olarak ne işe yarar?")

        # SQL / Veri
        if "sql" in combined_text or "veri" in combined_text or "data" in combined_text:
            pool.append("3. [Teknik] Soru: SQL'de 'LEFT JOIN' ile 'INNER JOIN' arasındaki fark nedir? Performans açısından hangisi ne zaman kullanılır?")

        # Web / Frontend
        if "html" in combined_text or "css" in combined_text or "javascript" in combined_text or "react" in combined_text:
            pool.append("1. [Teknik] Soru: Modern web tarayıcılarında 'Rendering' süreci nasıl işler? DOM ve Virtual DOM farkı nedir?")

        # AI / Machine Learning
        if "yapay zeka" in combined_text or "ai" in combined_text or "learning" in combined_text:
            pool.append("1. [Teknik] Soru: Overfitting (Aşırı Öğrenme) sorununu engellemek için model eğitiminde hangi teknikleri (Regularization, Dropout vb.) kullanırsınız?")
            pool.append("2. [Teknik] Soru: Bir sınıflandırma probleminde 'Precision' ve 'Recall' metrikleri arasındaki dengeyi (Trade-off) nasıl kurarsınız?")

        # Genel Sorular
        if not pool:
            pool = [
                "1. [Teknik] Soru: Şimdiye kadar karşılaştığınız en zor teknik problem neydi ve bunu nasıl çözdünüz?",
                "2. [Teknik] Soru: Bir projede kod kalitesini ve sürdürülebilirliği sağlamak için hangi prensipleri (SOLID, Clean Code) uygularsınız?",
                "3. [Teknik] Soru: Yeni bir teknolojiyi veya framework'ü öğrenirken nasıl bir yöntem izlersiniz?"
            ]

        # Davranışsal Sorular
        behavioral = [
            "4. [Davranışsal] Soru: Takım içinde bir arkadaşınızla teknik bir konuda anlaşmazlık yaşadığınızda bu durumu nasıl yönetirsiniz?",
            "5. [Davranışsal] Soru: Çok kısıtlı bir sürede (deadline) yetişmesi gereken bir proje üzerinde çalışırken baskı altında nasıl performans gösterirsiniz?"
        ]

        # Havuzdan rastgele veya sırayla 3 teknik soru seç
        import random
        selected_tech = pool[:3] # İlk 3 tanesini al

        # Formatı birleştir
        questions = "\n".join(selected_tech + behavioral)

    return {"questions": questions}
# Sunucuyu Başlat
nest_asyncio.apply()
def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

thread = threading.Thread(target=start_server, daemon=True)
thread.start()
print("FastAPI sunucusu (LEAKAGE FIX) başlatıldı!")

FastAPI sunucusu (LEAKAGE FIX) başlatıldı!


In [ ]:
# Cloudflare Tüneli Açma
time.sleep(3)
print("\n Cloudflare tüneli açılıyor...\n")

process = subprocess.Popen(
    ['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Linki yakala
time.sleep(5)
link_bulundu = False

try:
    for line in process.stderr:
        if "trycloudflare.com" in line:
            url_match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if url_match:
                public_url = url_match.group(0)
                print("\n" + "="*70)
                print(f" API LİNKİN: {public_url}")
                print(f" Endpoint: {public_url}/cevir")
                print("="*70 + "\n")
                link_bulundu = True
                break
except Exception as e:
    print(f"Link otomatik bulunamadı: {e}")

if not link_bulundu:
    print(" Link loglardan manuel kontrol et!")

# Sunucuyu açık tut
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Kapatılıyor...")